In [ ]:
# In this notebook, we train with all the distributions, test with the left out GMM
# Uses a range for the SNR, so training samples will have SNR in [0.01, 5]

In [ ]:
from trexselector_deep.data_loading import *
from trexselector_deep.generate_data import *
from trexselector_deep.metrics import *
from trexselector_deep.model import *
from trexselector_deep.training import *
from trexselector_deep.visualization import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generate Data

In [ ]:
# Change num_act!!!

T_stop_max = 5
L_factor = 1
n = 75
p = 150
train_data_dir = "/Users/arnau/Desktop/deeptrex/code/data/all_range.h5"
test_data_dir = "/Users/arnau/Desktop/deeptrex/code/data/(gmm_test)_N_systems=10000,SNR=1.0,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"

# Train Model

In [ ]:
input_size = get_phi_shape(train_data_dir)

hparams = {"num_epochs": 10,
           "batch_size": 16,
           "learning_rate": 0.001
           }

In [ ]:
lazy_loading = False
train_loader, test_loader = get_separate_data_loaders(train_data_dir, test_data_dir, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

model = FDPNet(input_size).to(device)
criterion = AsymmetricMSELoss(underestimation_weight=1.1)
optimizer = torch.optim.Adam(model.parameters(), lr=hparams["learning_rate"])

train_losses, test_losses = train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=hparams["num_epochs"], evaluate=True, device=device)

In [ ]:
plot_training_history(train_losses, test_losses)

# Save and Load

In [ ]:
save_model(model)
save_dataloaders(train_loader, test_loader)

In [ ]:
# train_data_dir = "data/(all FDP)_N_systems=1000,SNR=1.0,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=100"
# test_data_dir = "data/(all FDP)_N_systems=1000,SNR=1.0,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=100"
input_size = get_phi_shape(train_data_dir)
hparams = {"num_epochs": 10,
           "batch_size": 16,
           "learning_rate": 0.001
           }
lazy_loading = False

model = load_model(input_size)
train_loader, test_loader = get_separate_data_loaders(train_data_dir, test_data_dir, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

# Evaluate Model

In [ ]:
# Get FDP predictions for train and test data
train_loader_FDP, train_infer_FDP = get_loader_and_infer_FDP(train_loader, model, device=device)
test_loader_FDP, test_infer_FDP = get_loader_and_infer_FDP(test_loader, model, device=device)

In [ ]:
# Plot combined FDP histograms (actual vs predicted) for both train and test data
plot_combined_FDP(train_loader_FDP, train_infer_FDP, test_loader_FDP, test_infer_FDP)

In [ ]:
# Plot combined FDP overestimation and difference histograms in a single figure
plot_combined_overestimation_and_diff(train_loader_FDP, train_infer_FDP, test_loader_FDP, test_infer_FDP)

In [ ]:
alpha = 0.1
partial_train_loader = get_partial_loader(train_loader, 0.1)
partial_test_loader = get_partial_loader(test_loader, 0.01)

results = get_comparison_results(model, partial_train_loader, partial_test_loader, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)

# results = get_comparison_results(model, train_loader, test_loader, alpha=0.1, T_stop_max=T_stop_max, device=device, num_batches_to_group=10)

In [ ]:
# Plot optimal T_stop histograms
plot_optimal_T_stop_hist(results, loader_2=True, T_stop_max=T_stop_max)

In [ ]:
plot_optimal_v_T_stop_heatmap(results, loader_2=True, T_stop_max=T_stop_max, L=input_size[0])

In [ ]:
# Plot FDR and TPR comparisons
plot_FDR_TPR(results['loader_1']['model']['metrics'], results['loader_1']['trex'], 
             title_prefix="Train", alpha=alpha)
plot_FDR_TPR(results['loader_2']['model']['metrics'], results['loader_2']['trex'],
             title_prefix="Test", alpha=alpha)

In [ ]:
model_FDP1, real_FDP1, difference1 = plot_overlay_and_difference_FDP_heatmaps_3D(model, test_loader, T_stop_max=T_stop_max, alpha=0.05, L=input_size[0])

In [ ]:
model_FDP, real_FDP, difference = plot_overlay_and_difference_FDP_heatmaps_3D_interactive(model, test_loader, T_stop_max=T_stop_max, alpha=0.05, L=input_size[0])  # Add TRex boundry

# Multiple SNR levels

In [ ]:
alpha = 0.1
dummy_train_loader = get_partial_loader(train_loader, 0.001)

test_data_dir_snr001 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=0.01,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"
test_data_dir_snr02 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=0.2,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"
test_data_dir_snr04 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=0.4,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"
test_data_dir_snr06 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=0.6,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"
test_data_dir_snr08 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=0.8,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"
test_data_dir_snr1 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=1.0,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"
test_data_dir_snr2 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=2.0,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"
test_data_dir_snr5 = "/Users/arnau/Desktop/deeptrex/code/data/gmm_tests/(gmm_test)_N_systems=1000,SNR=5.0,T_stop_max=5,n=75,num_act=3,num_dummies=150,p=150,K=20,k_min=2,k_max=5,mean_min=-10.0,mean_max=10.0,std_min=0.5,std_max=2.0.h5"

## SNR=0.01

In [ ]:
_, test_loader_001 = get_separate_data_loaders(train_data_dir, test_data_dir_snr001, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_001 = get_comparison_results(model, dummy_train_loader, test_loader_001, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_001['loader_2']['model']['metrics'], results_001['loader_2']['trex'], title_prefix="SNR=0.01 Test", alpha=alpha)

## SNR=0.2

In [ ]:
_, test_loader_02 = get_separate_data_loaders(train_data_dir, test_data_dir_snr02, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_02 = get_comparison_results(model, dummy_train_loader, test_loader_02, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_02['loader_2']['model']['metrics'], results_02['loader_2']['trex'], title_prefix="SNR=0.2 Test", alpha=alpha)

## SNR=0.4

In [ ]:
_, test_loader_04 = get_separate_data_loaders(train_data_dir, test_data_dir_snr04, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_04 = get_comparison_results(model, dummy_train_loader, test_loader_04, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_04['loader_2']['model']['metrics'], results_04['loader_2']['trex'], title_prefix="SNR=0.4 Test", alpha=alpha)

## SNR=0.6

In [ ]:
_, test_loader_06 = get_separate_data_loaders(train_data_dir, test_data_dir_snr06, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_06 = get_comparison_results(model, dummy_train_loader, test_loader_06, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_06['loader_2']['model']['metrics'], results_06['loader_2']['trex'], title_prefix="SNR=0.6 Test", alpha=alpha)

## SNR=0.8

In [ ]:
_, test_loader_08 = get_separate_data_loaders(train_data_dir, test_data_dir_snr08, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_08 = get_comparison_results(model, dummy_train_loader, test_loader_08, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_08['loader_2']['model']['metrics'], results_08['loader_2']['trex'], title_prefix="SNR=0.8 Test", alpha=alpha)

## SNR=1

In [ ]:
_, test_loader_1 = get_separate_data_loaders(train_data_dir, test_data_dir_snr1, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_1 = get_comparison_results(model, dummy_train_loader, test_loader_1, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_1['loader_2']['model']['metrics'], results_1['loader_2']['trex'], title_prefix="SNR=1.0 Test", alpha=alpha)

## SNR=2

In [ ]:
_, test_loader_2 = get_separate_data_loaders(train_data_dir, test_data_dir_snr2, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_2 = get_comparison_results(model, dummy_train_loader, test_loader_2, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_2['loader_2']['model']['metrics'], results_2['loader_2']['trex'], title_prefix="SNR=2.0 Test", alpha=alpha)

## SNR=5

In [ ]:
_, test_loader_5 = get_separate_data_loaders(train_data_dir, test_data_dir_snr5, batch_size=hparams["batch_size"], lazy_loading=lazy_loading)

results_5 = get_comparison_results(model, dummy_train_loader, test_loader_5, alpha=alpha, T_stop_max=T_stop_max, L_max_factor=L_factor, device=device, num_batches_to_group=1)
plot_FDR_TPR(results_5['loader_2']['model']['metrics'], results_5['loader_2']['trex'], title_prefix="SNR=5.0 Test", alpha=alpha)

In [ ]:
import importlib
import trexselector_deep.data_loading
import trexselector_deep.generate_data
import trexselector_deep.metrics
import trexselector_deep.model
import trexselector_deep.training
import trexselector_deep.visualization

from trexselector_deep.data_loading import *
from trexselector_deep.generate_data import *
from trexselector_deep.metrics import *
from trexselector_deep.model import *
from trexselector_deep.training import *
from trexselector_deep.visualization import *

# Reload the module
importlib.reload(trexselector_deep.data_loading)
importlib.reload(trexselector_deep.generate_data)
importlib.reload(trexselector_deep.metrics)
importlib.reload(trexselector_deep.model)
importlib.reload(trexselector_deep.training)
importlib.reload(trexselector_deep.visualization)

# Now you can use the reloaded module
import trexselector_deep.data_loading
import trexselector_deep.generate_data
import trexselector_deep.metrics
import trexselector_deep.model
import trexselector_deep.training
import trexselector_deep.visualization

from trexselector_deep.data_loading import *
from trexselector_deep.generate_data import *
from trexselector_deep.metrics import *
from trexselector_deep.model import *
from trexselector_deep.training import *
from trexselector_deep.visualization import *